# Rag Retrieval Test Run

In [1]:
import json 
from sentence_transformers import SentenceTransformer 
import faiss 
import numpy as np

/Users/svenwu/Hustle/MachineLearning/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load synthetic data in.
with open("synthetic_student_reflections.json") as f:
  example_corpus = json.load(f)

In [3]:
f

<_io.TextIOWrapper name='synthetic_student_reflections.json' mode='r' encoding='UTF-8'>

In [4]:
print(f"Loaded {len(example_corpus)} exmaples.")

Loaded 25 exmaples.


### Embed Student Texts

In [5]:
# Load the embedder.
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embedder

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7811.33it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [6]:
# Get only the student_text from the JSON.
student_texts = [ex["student_text"] for ex in example_corpus]
student_texts[:3]

['I watched how my lab partner explained the experiment to everyone when we were confused. She made something complicated seem really simple, and I wish I could explain things like that.',
 'My group accidentally submitted the draft with all of our comments still visible. We were embarrassed at first, but looking back at some of the comments is actually pretty funny.',
 'I spent three hours working on the assignment and then found out the instructions had changed after I started. I wish someone had told us earlier because now I have to redo most of my work.']

In [7]:
# Embded all student_text entries.
student_text_embeddings = embedder.encode(student_texts, convert_to_numpy=True)
student_text_embeddings[:5]

array([[-0.08167081, -0.0350743 ,  0.04618319, ...,  0.1345141 ,
         0.04746348, -0.05028719],
       [-0.09006829, -0.02984722,  0.03510597, ...,  0.0344655 ,
        -0.07471598,  0.04870407],
       [-0.02693345,  0.05441548,  0.0054687 , ...,  0.09743757,
        -0.13128257, -0.04685029],
       [ 0.00585679,  0.01583065,  0.02454555, ...,  0.11912089,
        -0.10991519,  0.0521571 ],
       [-0.08291283, -0.01857998,  0.03763821, ..., -0.02041946,
        -0.02709852,  0.04316133]], shape=(5, 384), dtype=float32)

In [8]:
len(student_text_embeddings)

25

In [9]:
student_text_embeddings.shape

(25, 384)

### Build the FAISS Index

In [10]:
# Create an empty index — no data in it yet, just a container configured to hold vectors of a specific size.
index = faiss.IndexFlatL2(student_text_embeddings.shape[1])

In [11]:
index.add(student_text_embeddings)

In [12]:
print(f"Index built — {index.ntotal} vectors, dimension {student_text_embeddings.shape[1]}")

Index built — 25 vectors, dimension 384


In [13]:
def retrieve_similar(text, k=3):
  query_embedding = embedder.encode([text], convert_to_numpy=True)
  
  _, indices = index.search(query_embedding, k)
  return [example_corpus[i] for i in indices[0]]

# Helper function to display results.
def display_results(results):
    for r in results:
        print(f"\n* {r['student_text']}")

In [14]:
example_corpus[0]

{'id': 1,
 'student_text': 'I watched how my lab partner explained the experiment to everyone when we were confused. She made something complicated seem really simple, and I wish I could explain things like that.',
 'emotions': ['admiration'],
 'teacher_response': "It sounds like you really appreciated the way your partner approached the problem. Noticing effective communication in others can also help you identify skills you'd like to develop yourself."}

In [15]:
test_text = example_corpus[0]["student_text"]
test_text

'I watched how my lab partner explained the experiment to everyone when we were confused. She made something complicated seem really simple, and I wish I could explain things like that.'

In [16]:
results = results = retrieve_similar(test_text, k=3)
display_results(results)


* I watched how my lab partner explained the experiment to everyone when we were confused. She made something complicated seem really simple, and I wish I could explain things like that.

* I don't understand why the experiment produced such a different result from what we expected. At first I thought we made a mistake, but now I'm wondering if there's something interesting happening that we haven't considered.

* I read the directions three times and still don't understand what we're supposed to include in the final section. I know I'm missing something, but I can't figure out what it is.
